# Deep-Dive: Strictly Conserved Metastatic Metabolic Signature
## Exploring STAT3, Oxygen Gradients, Directional CCC, and Clinical Prognosis
This notebook addresses **Priority 1** and **Priority 2** next steps from the AI Summary (Version 5).
Specifically, we cover:
1. **STAT3 Regulatory Network Reconstruction** (Step 1)
2. **Intratumoural Oxygen Gradient Simulation** (Step 2)
3. **Directionality-Aware Metabolic Communication Scoring** (Step 4)
4. **Metastatic Signature Score TCGA Clinical Validation** (Step 5)


In [1]:
import os
import sys
import pandas as pd
import subprocess

# Add scripts dir to path
sys.path.append(os.path.abspath("."))
import sys
if '..' not in sys.path: sys.path.append('..')
from pan_cancer_config import ANALYSIS_SUFFIX


### Step 1: STAT3 Regulatory Network Reconstruction
**Purpose**: Map all strictly conserved pan-cancer conserved genes within the STAT3 transcriptional network using ChEA 2022 ChIP-Seq data.
**Interpretation**: Validates STAT3 as an upstream regulatory master switch for the metastatic metabolic signature.


In [2]:
from compute_stat3_network import compute_stat3_network
base_dir = '..' if os.path.basename(os.getcwd()) == 'scripts' else '.'

compute_stat3_network()
# Load output
output_base=f"deepdive_conserved_metabGeneSig{ANALYSIS_SUFFIX}"
output_filename =  "stat3_u87_targets_strictly_conserved.csv"
stat3_out = os.path.join(base_dir, "output", output_base, "stat3_network",output_filename)
if os.path.exists(stat3_out):
    df = pd.read_csv(stat3_out)
    display(df)


In [3]:
import networkx as nx
import matplotlib.pyplot as plt

if os.path.exists(stat3_out):
    G = nx.from_pandas_edgelist(df, 'Source', 'Target')
    plt.figure(figsize=(8,6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_nodes(G, pos, nodelist=['STAT3'], node_color='red', node_size=1000)
    nx.draw_networkx_nodes(G, pos, nodelist=[n for n in G.nodes if n != 'STAT3'], node_color='lightblue', node_size=500)
    nx.draw_networkx_edges(G, pos, alpha=0.5)
    nx.draw_networkx_labels(G, pos)
    plt.title("STAT3 Transcriptional Network of Metastatic Metabolic Genes")
    plt.axis('off')
    plt.show()


### Step 2: Intratumoural Oxygen Gradient Simulation
**Purpose**: Reconstruct oxygen gradients using hypoxia signature proxy scores (VEGFA, SLC2A1, BNIP3) from established HIF-1α literature (Semenza et al.) and project the conserved-gene score onto this gradient.
**Interpretation**: Determines if the pre-metastatic subclone maps to the hypoxic core of the primary tumor.


In [4]:
from simulate_oxygen_gradient import simulate_oxygen_gradient
simulate_oxygen_gradient()


In [5]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Robust path resolution
base_dir = '..' if os.path.basename(os.getcwd()) == 'scripts' else '.'
oxy_dir = os.path.join(base_dir, "output", output_base, "oxygen_gradient")

oxy_files = glob.glob(os.path.join(oxy_dir, "*_primary_oxygen_gradient_scores.csv"))

if oxy_files:
    n_files = len(oxy_files)
    fig, axes = plt.subplots(1, n_files, figsize=(6 * n_files, 5), sharey=True)
    if n_files == 1:
        axes = [axes]
        
    for ax, oxy_file in zip(axes, oxy_files):
        cancer_name = os.path.basename(oxy_file).split('_')[0].capitalize()
        df = pd.read_csv(oxy_file)
        
        sns.regplot(
            data=df, x='hypoxia_score', y='metastatic_score',
            scatter_kws={'alpha': 0.5, 's': 20, 'color': 'gray'},
            line_kws={'color': 'crimson'}, ax=ax
        )
        
        r, p = pearsonr(df['hypoxia_score'], df['metastatic_score'])
        ax.set_title(f"{cancer_name} Primary\n(r={r:.2f}, p={p:.2e})", fontsize=12)
        ax.set_xlabel('Simulated Oxygen Gradient', fontsize=10)
        if ax == axes[0]:
            ax.set_ylabel('Strictly Conserved Metastatic Score', fontsize=10)
        else:
            ax.set_ylabel('')
        
    plt.tight_layout()
    plt.show()
else:
    print(f"No oxygen gradient results found in {oxy_dir}")

### Step 4: Directionality-Aware Metabolic Communication Scoring
**Purpose**: Separate enzymes into 'producing' and 'consuming' classes based on MetalinksDB reaction stoichiometry and score metabolic cross-talk directionally.


In [6]:
from compute_directional_ccc import compute_directional_ccc
compute_directional_ccc()


In [7]:
import os
base_dir = '..' if os.path.basename(os.getcwd()) == 'scripts' else '.'
from matplotlib.ticker import MultipleLocator
ccc_file = os.path.join(
    base_dir,
    "output",
    output_base, "directional_ccc",
    "metalinks_direction_classes.csv"
)

if os.path.exists(ccc_file):
    ccc_df = pd.read_csv(ccc_file)

    plt.figure(figsize=(5,5))

    ax = sns.countplot(
        data=ccc_df,
        x='Direction_Class',
        palette='Set2'
    )
    # Force y-axis ticks every 5
    ax.yaxis.set_major_locator(MultipleLocator(5))

    plt.title('Directional Metabolic Communication Classes')
    plt.ylabel('Number of Genes')
    plt.xlabel('Direction Class')

    # Add count labels on bars
    for p in ax.patches:
        height = int(p.get_height())

        ax.annotate(
            f'{height}',
            (
                p.get_x() + p.get_width() / 2,
                height
            ),
            ha='center',
            va='bottom',
            fontsize=10,
            xytext=(0, 3),
            textcoords='offset points'
        )

    plt.tight_layout()
    plt.show()

### Step 5: TCGA Survival Validation
**Purpose**: Validate the conserved-gene Metastatic Metabolic Score on TCGA primary tumor data against distant metastasis-free survival.


In [8]:
from validate_tcga_signature import validate_tcga_signature
validate_tcga_signature()


### Step 5.1: Empirical Validation (Permutation Test)
**Purpose**: To verify that our conserved-gene signature's predictive power is statistically significantly better than randomly chosen genes, we generate 1000 random conserved-gene signatures, compute their Hazard Ratios, and plot our true signature against this null distribution.


In [9]:
from compute_permutation_null import compute_permutation_null
# Compute the null distribution (this reads the TCGA file and iterates Cox models)
compute_permutation_null(n_permutations=1000)


In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
base_dir = '..' if os.path.basename(os.getcwd()) == 'scripts' else '.'


null_file = os.path.join(base_dir, "output", output_base, "tcga_validation", "null_distribution_metrics.csv")
true_file = os.path.join(base_dir, "output", output_base, "tcga_validation", "true_signature_metrics.csv")

if os.path.exists(null_file) and os.path.exists(true_file):
    null_df = pd.read_csv(null_file)
    true_df = pd.read_csv(true_file)
    
    # Plot true HR vs null distribution for each cancer
    cancers = true_df['TCGA_Cohort'].unique()
    fig, axes = plt.subplots(len(cancers), 1, figsize=(8, 4 * len(cancers)))
    if len(cancers) == 1:
        axes = [axes]
        
    for ax, cancer in zip(axes, cancers):
        cancer_null = null_df[null_df['TCGA_Cohort'] == cancer]['Hazard_Ratio']
        cancer_true = true_df[true_df['TCGA_Cohort'] == cancer]['Hazard_Ratio'].iloc[0]
        
        sns.histplot(cancer_null, bins=20, kde=True, color='lightgray', ax=ax, label=f'Random Gene Signatures (Null)')
        ax.axvline(cancer_true, color='red', linestyle='--', linewidth=2, label=f'True Signature (HR={cancer_true:.2f})')
        
        ax.set_title(f'TCGA-{cancer}: True Signature vs. Null Distribution')
        ax.set_xlabel('Hazard Ratio (High vs Low Risk)')
        ax.set_ylabel('Frequency')
        ax.legend()
        
    plt.tight_layout()
    plt.show()


In [12]:
import subprocess
import sys
import os
base_dir = '..' if os.path.basename(os.getcwd()) == 'scripts' else '.'


notebook_filename = 'deepdive_conserved_metabGeneSig.ipynb'
output_filename =output_filename.replace(".csv","")
output_dir = os.path.join(base_dir, 'output', output_base)
os.makedirs(output_dir, exist_ok=True)

cmd_html = [sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", notebook_filename, "--output-dir", output_dir, "--output", output_base]
res_html = subprocess.run(cmd_html, capture_output=True, text=True)

if res_html.returncode == 0:
    print(f"🎉 SUCCESS: Notebook successfully exported to '{os.path.join(output_dir, output_base, output_filename)}.html'")
else:
    print("❌ HTML export failed.")
    print(res_html.stderr)
